In [1]:
# ========================================
# AfyaMetrix Data Simulation Engine
# Notebook: 01_data_simulation.ipynb
# Author: Omosomi Ann Hassan (AI/ML Engineer)
# ========================================

# pandas: handles our data in table format (like Excel but in Python)
import pandas as pd

# numpy: powers all our math and random number generation
import numpy as np

# datetime: helps us work with dates and time ranges
from datetime import datetime, timedelta

# os: lets us interact with the file system (saving files to folders)
import os

# We set a random seed so our "random" data is reproducible
# This means every time you run this, you get the SAME data
# which is important for consistency across your team
np.random.seed(42)

print("✅ Imports successful")
print("📊 Building AfyaMetrix Data Simulation Engine...")

✅ Imports successful
📊 Building AfyaMetrix Data Simulation Engine...


In [2]:
# ========================================
# DEFINING OUR PAN-AFRICA GEOGRAPHY
# ========================================
# We define 10 countries across different African regions
# Each country has sub-regions (like counties/states)
# This reflects real African health system structures

AFRICA_REGIONS = {
    # East Africa
    "Kenya": {
        "regions": ["Nairobi", "Kisumu", "Mombasa", "Nakuru", "Eldoret", 
                    "Garissa", "Turkana", "Mandera"],
        "population_base": 54000000,
        "climate": "mixed",        # affects which diseases dominate
        "healthcare_access": 0.65  # 65% of population has reasonable access
    },
    "Ethiopia": {
        "regions": ["Addis Ababa", "Oromia", "Amhara", "Tigray", 
                    "Afar", "Somali Region", "Sidama"],
        "population_base": 120000000,
        "climate": "mixed",
        "healthcare_access": 0.45
    },
    "Uganda": {
        "regions": ["Kampala", "Gulu", "Mbarara", "Jinja", 
                    "Mbale", "Arua", "Lira"],
        "population_base": 47000000,
        "climate": "tropical",
        "healthcare_access": 0.55
    },
    
    # West Africa
    "Nigeria": {
        "regions": ["Lagos", "Kano", "Abuja", "Rivers", 
                    "Oyo", "Borno", "Kaduna", "Anambra"],
        "population_base": 220000000,
        "climate": "tropical",
        "healthcare_access": 0.50
    },
    "Ghana": {
        "regions": ["Accra", "Kumasi", "Tamale", "Sekondi", 
                    "Sunyani", "Northern Region", "Upper East"],
        "population_base": 33000000,
        "climate": "tropical",
        "healthcare_access": 0.70
    },
    "Senegal": {
        "regions": ["Dakar", "Thiès", "Diourbel", "Saint-Louis", 
                    "Ziguinchor", "Tambacounda"],
        "population_base": 17000000,
        "climate": "sahel",
        "healthcare_access": 0.60
    },
    
    # Southern Africa
    "Tanzania": {
        "regions": ["Dar es Salaam", "Dodoma", "Mwanza", "Arusha", 
                    "Mbeya", "Morogoro", "Zanzibar"],
        "population_base": 63000000,
        "climate": "tropical",
        "healthcare_access": 0.55
    },
    "Zambia": {
        "regions": ["Lusaka", "Copperbelt", "Southern Province", 
                    "Eastern Province", "Northern Province"],
        "population_base": 19000000,
        "climate": "mixed",
        "healthcare_access": 0.50
    },
    
    # North Africa
    "Sudan": {
        "regions": ["Khartoum", "Omdurman", "Darfur", "Kassala", 
                    "Blue Nile", "Red Sea State"],
        "population_base": 45000000,
        "climate": "arid",
        "healthcare_access": 0.35
    },
    
    # Central Africa
    "DRC": {
        "regions": ["Kinshasa", "Lubumbashi", "Goma", "Bukavu", 
                    "Kisangani", "Mbuji-Mayi", "Kananga"],
        "population_base": 100000000,
        "climate": "tropical",
        "healthcare_access": 0.30
    }
}

print(f"✅ Defined {len(AFRICA_REGIONS)} countries")
total_regions = sum(len(v['regions']) for v in AFRICA_REGIONS.values())
print(f"✅ Total regions/sub-counties: {total_regions}")

✅ Defined 10 countries
✅ Total regions/sub-counties: 68


In [3]:
# ========================================
# DISEASE DEFINITIONS
# ========================================
# Each disease has properties that affect how we simulate it
# These are based on real epidemiological characteristics

DISEASES = {
    "Malaria": {
        "base_rate": 45,          # average daily cases per 100k population
        "seasonality": "rainy",   # spikes during rainy season
        "outbreak_probability": 0.03,  # 3% chance of outbreak on any given day
        "outbreak_multiplier": 4.0,    # cases multiply by 4x during outbreak
        "climates": ["tropical", "mixed"],  # which climates it dominates in
        "tier": 1
    },
    "Cholera": {
        "base_rate": 8,
        "seasonality": "rainy",
        "outbreak_probability": 0.02,
        "outbreak_multiplier": 6.0,    # cholera spreads very fast
        "climates": ["tropical", "mixed", "arid"],
        "tier": 1
    },
    "Tuberculosis": {
        "base_rate": 12,
        "seasonality": "none",    # TB has no strong seasonal pattern
        "outbreak_probability": 0.01,
        "outbreak_multiplier": 2.0,
        "climates": ["tropical", "mixed", "arid", "sahel"],
        "tier": 1
    },
    "Meningitis": {
        "base_rate": 3,
        "seasonality": "dry",     # meningitis spikes in dry/harmattan season
        "outbreak_probability": 0.015,
        "outbreak_multiplier": 5.0,
        "climates": ["sahel", "arid"],
        "tier": 1
    },
    "Typhoid": {
        "base_rate": 15,
        "seasonality": "rainy",
        "outbreak_probability": 0.02,
        "outbreak_multiplier": 3.0,
        "climates": ["tropical", "mixed", "sahel"],
        "tier": 2
    },
    "Mpox": {
        "base_rate": 2,
        "seasonality": "none",
        "outbreak_probability": 0.01,
        "outbreak_multiplier": 7.0,    # very high multiplier — rare but explosive
        "climates": ["tropical"],
        "tier": 2
    },
    "Dengue": {
        "base_rate": 6,
        "seasonality": "rainy",
        "outbreak_probability": 0.015,
        "outbreak_multiplier": 4.0,
        "climates": ["tropical", "mixed"],
        "tier": 2
    },
    "Respiratory_Infections": {
        "base_rate": 35,
        "seasonality": "dry",
        "outbreak_probability": 0.025,
        "outbreak_multiplier": 3.0,
        "climates": ["tropical", "mixed", "arid", "sahel"],
        "tier": 3
    },
    "Diarrheal_Disease": {
        "base_rate": 40,
        "seasonality": "rainy",
        "outbreak_probability": 0.02,
        "outbreak_multiplier": 3.5,
        "climates": ["tropical", "mixed", "arid", "sahel"],
        "tier": 3
    },
    "Malnutrition": {
        "base_rate": 20,
        "seasonality": "none",
        "outbreak_probability": 0.005,
        "outbreak_multiplier": 2.0,
        "climates": ["arid", "sahel"],
        "tier": 3
    }
}

print(f"✅ Defined {len(DISEASES)} diseases")
print("\nDisease Tiers:")
for name, props in DISEASES.items():
    print(f"  Tier {props['tier']}: {name} — base rate: {props['base_rate']} cases/100k/day")

✅ Defined 10 diseases

Disease Tiers:
  Tier 1: Malaria — base rate: 45 cases/100k/day
  Tier 1: Cholera — base rate: 8 cases/100k/day
  Tier 1: Tuberculosis — base rate: 12 cases/100k/day
  Tier 1: Meningitis — base rate: 3 cases/100k/day
  Tier 2: Typhoid — base rate: 15 cases/100k/day
  Tier 2: Mpox — base rate: 2 cases/100k/day
  Tier 2: Dengue — base rate: 6 cases/100k/day
  Tier 3: Respiratory_Infections — base rate: 35 cases/100k/day
  Tier 3: Diarrheal_Disease — base rate: 40 cases/100k/day
  Tier 3: Malnutrition — base rate: 20 cases/100k/day


In [4]:
# ========================================
# CORE SIMULATION ENGINE
# ========================================

def get_seasonal_multiplier(date, seasonality, country):
    """
    Returns a multiplier based on the time of year.
    Rainy season in East/West Africa: March-May, October-December
    Dry season: June-September, January-February
    """
    month = date.month
    
    if seasonality == "none":
        return 1.0  # no seasonal effect
    
    elif seasonality == "rainy":
        # Cases spike during rainy months
        if month in [3, 4, 5, 10, 11, 12]:
            return np.random.uniform(1.5, 2.5)  # 1.5x to 2.5x more cases
        else:
            return np.random.uniform(0.6, 1.0)  # fewer cases in dry season
    
    elif seasonality == "dry":
        # Cases spike during dry months (meningitis, respiratory)
        if month in [1, 2, 6, 7, 8, 9]:
            return np.random.uniform(1.5, 2.2)
        else:
            return np.random.uniform(0.7, 1.0)
    
    return 1.0


def get_urban_rural_factor(region, country):
    """
    Urban areas report more cases (better detection)
    but rural areas have worse outcomes.
    We simulate this by giving urban regions higher reported counts.
    """
    # These are the major urban centers in our dataset
    urban_centers = [
        "Nairobi", "Lagos", "Accra", "Dakar", "Kinshasa",
        "Addis Ababa", "Kampala", "Dar es Salaam", "Lusaka",
        "Khartoum", "Nairobi", "Abuja", "Mombasa"
    ]
    
    if region in urban_centers:
        return np.random.uniform(1.3, 1.8)  # urban: higher reporting
    else:
        return np.random.uniform(0.4, 0.9)  # rural: lower reporting (under-reported)


def simulate_outbreak(current_cases, outbreak_multiplier, duration_remaining):
    """
    Simulates an active outbreak.
    Cases spike at start, then gradually decline.
    This creates the characteristic 'outbreak curve' shape.
    """
    # Outbreak intensity decreases as it progresses
    intensity = max(0.3, duration_remaining / 14)  # 14-day average outbreak
    spike = current_cases * outbreak_multiplier * intensity
    
    # Add noise so it doesn't look perfectly mathematical
    noise = np.random.normal(0, spike * 0.1)
    
    return max(0, int(spike + noise))


def generate_health_data(start_date="2023-01-01", end_date="2024-12-31"):
    """
    Main simulation function.
    Generates daily case counts for every region, country, and disease
    across the specified date range.
    
    Returns a pandas DataFrame — the core dataset for all our models.
    """
    
    print("🔄 Starting data simulation...")
    print(f"   Date range: {start_date} to {end_date}")
    
    # Convert string dates to datetime objects
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    
    # Generate every day in our range
    date_range = []
    current_date = start
    while current_date <= end:
        date_range.append(current_date)
        current_date += timedelta(days=1)
    
    print(f"   Total days to simulate: {len(date_range)}")
    
    # This will hold all our rows of data
    records = []
    
    # Track active outbreaks: {(country, region, disease): days_remaining}
    active_outbreaks = {}
    
    # Loop through every country
    for country, country_data in AFRICA_REGIONS.items():
        regions = country_data["regions"]
        climate = country_data["climate"]
        healthcare_access = country_data["healthcare_access"]
        population_base = country_data["population_base"]
        
        # Population per region (roughly equal split with some variance)
        region_population = population_base / len(regions)
        
        # Loop through every region in this country
        for region in regions:
            
            urban_factor = get_urban_rural_factor(region, country)
            
            # Loop through every disease
            for disease, disease_data in DISEASES.items():
                
                # Skip diseases that don't match this climate
                # (Meningitis is rare in tropical regions, for example)
                if climate not in disease_data["climates"]:
                    # Still include it but with very low base rate
                    adjusted_base = disease_data["base_rate"] * 0.1
                else:
                    adjusted_base = disease_data["base_rate"]
                
                outbreak_key = (country, region, disease)
                
                # Loop through every day
                for date in date_range:
                    
                    # --- CHECK FOR NEW OUTBREAK ---
                    if outbreak_key not in active_outbreaks:
                        if np.random.random() < disease_data["outbreak_probability"]:
                            # Outbreak starts! Lasts 7-21 days
                            active_outbreaks[outbreak_key] = np.random.randint(7, 21)
                    
                    # --- CALCULATE BASE CASES ---
                    seasonal_mult = get_seasonal_multiplier(
                        date, disease_data["seasonality"], country
                    )
                    
                    # Base cases per day for this region
                    # Formula: (base_rate / 100000) * population * seasonal * urban
                    base_cases = (
                        adjusted_base / 100000 * 
                        region_population * 
                        seasonal_mult * 
                        urban_factor * 
                        healthcare_access  # lower access = fewer reported cases
                    )
                    
                    # Add natural random noise (real data is never perfectly smooth)
                    noise = np.random.normal(0, base_cases * 0.15)
                    daily_cases = max(0, int(base_cases + noise))
                    
                    # --- APPLY OUTBREAK MULTIPLIER IF ACTIVE ---
                    is_outbreak = False
                    if outbreak_key in active_outbreaks:
                        days_left = active_outbreaks[outbreak_key]
                        daily_cases = simulate_outbreak(
                            daily_cases, 
                            disease_data["outbreak_multiplier"],
                            days_left
                        )
                        is_outbreak = True
                        
                        # Count down the outbreak
                        active_outbreaks[outbreak_key] -= 1
                        if active_outbreaks[outbreak_key] <= 0:
                            del active_outbreaks[outbreak_key]
                    
                    # --- STORE THE RECORD ---
                    records.append({
                        "date": date.strftime("%Y-%m-%d"),
                        "country": country,
                        "region": region,
                        "disease": disease,
                        "cases": daily_cases,
                        "is_outbreak": is_outbreak,
                        "population": int(region_population),
                        "healthcare_access": healthcare_access,
                        "climate": climate,
                        "urban_factor": round(urban_factor, 3),
                        "seasonal_multiplier": round(seasonal_mult, 3)
                    })
    
    # Convert list of records to a DataFrame
    df = pd.DataFrame(records)
    
    print(f"\n✅ Simulation complete!")
    print(f"   Total records generated: {len(df):,}")
    print(f"   Countries: {df['country'].nunique()}")
    print(f"   Regions: {df['region'].nunique()}")
    print(f"   Diseases: {df['disease'].nunique()}")
    print(f"   Date range: {df['date'].min()} to {df['date'].max()}")
    print(f"   Outbreak events: {df['is_outbreak'].sum():,}")
    
    return df

# Run it
df_raw = generate_health_data()

🔄 Starting data simulation...
   Date range: 2023-01-01 to 2024-12-31
   Total days to simulate: 731

✅ Simulation complete!
   Total records generated: 497,080
   Countries: 10
   Regions: 68
   Diseases: 10
   Date range: 2023-01-01 to 2024-12-31
   Outbreak events: 89,805


In [5]:
# ========================================
# SAVE AND PREVIEW
# ========================================

# Save to your data/raw folder
save_path = r"C:\Users\hassa\afyametrix\data\raw\africa_health_simulated.csv"
df_raw.to_csv(save_path, index=False)
print(f"✅ Dataset saved to: {save_path}")

# Preview the first few rows
print("\n📊 Sample of your data:")
print(df_raw.head(10).to_string())

# Quick statistics
print("\n📈 Cases summary by disease:")
print(df_raw.groupby("disease")["cases"].agg(["mean", "max", "sum"]).round(1))

✅ Dataset saved to: C:\Users\hassa\afyametrix\data\raw\africa_health_simulated.csv

📊 Sample of your data:
         date country   region  disease  cases  is_outbreak  population  healthcare_access climate  urban_factor  seasonal_multiplier
0  2023-01-01   Kenya  Nairobi  Malaria   2184        False     6750000               0.65   mixed         1.487                0.893
1  2023-01-02   Kenya  Nairobi  Malaria   1917        False     6750000               0.65   mixed         1.487                0.623
2  2023-01-03   Kenya  Nairobi  Malaria   2252        False     6750000               0.65   mixed         1.487                0.840
3  2023-01-04   Kenya  Nairobi  Malaria   1952        False     6750000               0.65   mixed         1.487                0.722
4  2023-01-05   Kenya  Nairobi  Malaria   2547        False     6750000               0.65   mixed         1.487                0.773
5  2023-01-06   Kenya  Nairobi  Malaria   1623        False     6750000               0.6